In [1]:
!pip install datasets torch -q

In [2]:
import re
import math
import torch
import torch.nn as nn
import torch.optim as optim

from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader

# 1. Cargar dataset de frases

In [3]:
dataset = load_dataset(
    "agentlans/high-quality-english-sentences",
    split="train[:1000]"
)

sentences = [row["text"] for row in dataset]

sentences[:5]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/2.30k [00:00<?, ?B/s]

train.txt.gz:   0%|          | 0.00/85.5M [00:00<?, ?B/s]

test.txt.gz:   0%|          | 0.00/9.49M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1534699 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/170522 [00:00<?, ? examples/s]

['Soon we dropped into a living forest, where cold-tolerant evergreens and boreal animals still evoke the Canadian heritage of an ecosystem pushed south by glaciers 20,000 years ago.',
 'Annual population growth rate (2011 est., CIA World Factbook): 1.284%.',
 'This has led to the recent banning of Neonics in the EU, however the US and Canada are still using this chemical pesticide.',
 "In addition, these colors weren't confined to a province but rather irregularly scattered across various regions over all of China.",
 'A family member or a support person may stay with a patient during recovery.']

# 2. Limpieza del texto

In [4]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


clean_sentences = [clean_text(sentence) for sentence in sentences]
clean_sentences = [s for s in clean_sentences if len(s.split()) >= 5]

clean_sentences[:5]

['soon we dropped into a living forest where coldtolerant evergreens and boreal animals still evoke the canadian heritage of an ecosystem pushed south by glaciers 20000 years ago',
 'annual population growth rate 2011 est cia world factbook 1284',
 'this has led to the recent banning of neonics in the eu however the us and canada are still using this chemical pesticide',
 'in addition these colors werent confined to a province but rather irregularly scattered across various regions over all of china',
 'a family member or a support person may stay with a patient during recovery']

# 3. Crear vocabulario

Usaremos tokens especiales:

- `<PAD>`: relleno.
- `<OOV>`: palabra desconocida.

In [5]:
tokenized_sentences = [sentence.split() for sentence in clean_sentences]

vocab = {
    "<PAD>": 0,
    "<OOV>": 1
}

for sentence in tokenized_sentences:
    for word in sentence:
        if word not in vocab:
            vocab[word] = len(vocab)

index_to_word = {idx: word for word, idx in vocab.items()}

vocab_size = len(vocab)

print("Tamaño del vocabulario:", vocab_size)

Tamaño del vocabulario: 6194


In [6]:
def text_to_sequence(sentence):
    return [vocab.get(word, vocab["<OOV>"]) for word in sentence.split()]


sequences = [text_to_sequence(sentence) for sentence in clean_sentences]

In [8]:
max_len = 20

input_data = []
target_data = []

for seq in sequences:
    if len(seq) < 2:
        continue

    seq = seq[:max_len]

    input_seq = seq[:-1]
    target_seq = seq[1:]

    input_data.append(input_seq)
    target_data.append(target_seq)

print("Cantidad de ejemplos:", len(input_data))

Cantidad de ejemplos: 992


In [9]:
def pad_sequence(seq, max_len):
    return seq + [vocab["<PAD>"]] * (max_len - len(seq))


input_data = [pad_sequence(seq, max_len - 1) for seq in input_data]
target_data = [pad_sequence(seq, max_len - 1) for seq in target_data]

X = torch.tensor(input_data, dtype=torch.long)
y = torch.tensor(target_data, dtype=torch.long)

print("Forma de X:", X.shape)
print("Forma de y:", y.shape)

Forma de X: torch.Size([992, 19])
Forma de y: torch.Size([992, 19])


# 5. Dataset y DataLoader

In [10]:
class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        return self.X[index], self.y[index]


train_dataset = TextDataset(X, y)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

# TRANSFORMERS

# 6. Positional Encoding

El Transformer no procesa la secuencia palabra por palabra como una RNN.

Por eso necesita información de posición para saber el orden de las palabras.

In [11]:
class PositionalEncoding(nn.Module):
    def __init__(self, embedding_dim, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, embedding_dim)

        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, embedding_dim, 2).float()
            * (-math.log(10000.0) / embedding_dim)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)

        self.register_buffer("pe", pe)

    def forward(self, x):
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len]

# 7. Máscara causal

La máscara causal evita que el modelo vea palabras futuras.

Por ejemplo, para predecir la palabra 3, solo puede ver las palabras 1 y 2.

In [12]:
def generate_causal_mask(size):
    mask = torch.triu(torch.ones(size, size), diagonal=1)
    mask = mask.masked_fill(mask == 1, float("-inf"))
    return mask

# 8. Modelo Transformer para lenguaje

Este modelo tiene:

- Embedding
- Positional Encoding
- Transformer Encoder
- Capa lineal final

In [13]:
class TransformerLanguageModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim=128,
        num_heads=4,
        hidden_dim=256,
        num_layers=2,
        dropout=0.2
    ):
        super().__init__()

        self.embedding_dim = embedding_dim

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        self.positional_encoding = PositionalEncoding(
            embedding_dim=embedding_dim
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim,
            dropout=dropout,
            batch_first=True
        )

        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.fc = nn.Linear(embedding_dim, vocab_size)

    def forward(self, x):
        seq_len = x.size(1)

        causal_mask = generate_causal_mask(seq_len).to(x.device)

        x = self.embedding(x) * math.sqrt(self.embedding_dim)

        x = self.positional_encoding(x)

        transformer_output = self.transformer_encoder(
            x,
            mask=causal_mask
        )

        logits = self.fc(transformer_output)

        return logits

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TransformerLanguageModel(
    vocab_size=vocab_size,
    embedding_dim=128,
    num_heads=4,
    hidden_dim=256,
    num_layers=2,
    dropout=0.2
).to(device)

model

TransformerLanguageModel(
  (embedding): Embedding(6194, 128, padding_idx=0)
  (positional_encoding): PositionalEncoding()
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=256, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
        (linear2): Linear(in_features=256, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.2, inplace=False)
        (dropout2): Dropout(p=0.2, inplace=False)
      )
    )
  )
  (fc): Linear(in_features=128, out_features=6194, bias=True)
)

# 9. Pérdida y optimizador

Usamos `CrossEntropyLoss`.

Ignoramos el token `<PAD>` para que el modelo no sea penalizado por los rellenos.

In [15]:
criterion = nn.CrossEntropyLoss(ignore_index=vocab["<PAD>"])
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 10. Entrenamiento

In [16]:
epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch_X, batch_y in train_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()

        outputs = model(batch_X)

        outputs = outputs.reshape(-1, vocab_size)
        batch_y = batch_y.reshape(-1)

        loss = criterion(outputs, batch_y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(f"Epoch [{epoch+1}/{epochs}] - Loss: {avg_loss:.4f}")

Epoch [1/10] - Loss: 8.6074
Epoch [2/10] - Loss: 7.7861
Epoch [3/10] - Loss: 7.2956
Epoch [4/10] - Loss: 7.0403
Epoch [5/10] - Loss: 6.8960
Epoch [6/10] - Loss: 6.7590
Epoch [7/10] - Loss: 6.6426
Epoch [8/10] - Loss: 6.5074
Epoch [9/10] - Loss: 6.3650
Epoch [10/10] - Loss: 6.2131


# 11. Predecir la siguiente palabra

In [17]:
def predict_next_word(seed_text):
    model.eval()

    seed_text = clean_text(seed_text)

    seq = text_to_sequence(seed_text)
    seq = seq[-(max_len - 1):]

    input_seq = pad_sequence(seq, max_len - 1)

    input_tensor = torch.tensor([input_seq], dtype=torch.long).to(device)

    with torch.no_grad():
        output = model(input_tensor)

        last_position = len(seq) - 1
        logits = output[0, last_position]

        predicted_index = torch.argmax(logits).item()

    return index_to_word.get(predicted_index, "<OOV>")

In [18]:
predict_next_word("machine learning is")

'the'

# 12. Completar una oración

In [19]:
def complete_sentence(seed_text, next_words=10):
    result = seed_text

    for _ in range(next_words):
        next_word = predict_next_word(result)
        result += " " + next_word

    return result

In [20]:
complete_sentence("machine learning is", next_words=8)

'machine learning is the same of the first of the first'

In [21]:
complete_sentence("the future of technology", next_words=8)

'the future of technology and the first to the first and the'

In [22]:
complete_sentence("artificial intelligence can", next_words=8)

'artificial intelligence can and the first to the first of the'